# Pickup Too Soon (Infleeting) — Root Cause Analysis

**Period:** Last 3 months (Dec 2025 – Mar 2026)

**Critical discovery:** `pickup_too_soon` is created by TWO separate systems:
1. **`check-infleeting-blockers.ts`** — cars in statuses `ORDERED` through `READY_TO_DELIVER`. This is what we care about.
2. **`check-return-blockers.ts`** — in-subscription cars nearing end of subscription (return pickup within 24h). This is routine/transient noise.

We must filter to **infleeting-only** by checking car status at time of PTS.

**Key questions:**
1. Of infleeting PTS cars, how many had co-active blockers?
2. For those without — what happened? (blocker just resolved?)
3. Were blockers already active when the car hit `READY_TO_DELIVER`?
4. Why are we setting RTD so loosely?

**Code context:** `update-car-lifecycle-state.ts:122-124` transitions to RTD when `TECH_PREP_DONE` + `REGISTERED`. No blocker checks.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from google.cloud import bigquery

pio.renderers.default = 'svg'

client = bigquery.Client(project='datawarehouse-304513')

def run_query(query):
    return client.query(query).to_dataframe()

# Date range: last 3 months
DATE_FROM = '2025-12-01'
DATE_TO = '2026-03-10'
print(f'Analysis period: {DATE_FROM} to {DATE_TO}')

## 1. Infleeting PTS Population & Co-Active Blockers

Filter to cars that were in infleeting statuses (not `in_subscription`) when PTS fired.
Then check which other blockers were still active at that moment.

In [ ]:
# Step 1: Identify infleeting PTS cars (exclude in_subscription = return blocker noise)
infleeting_pts_query = f"""
WITH
pts_events AS (
  SELECT car_id, MIN(time) AS pts_time
  FROM `datawarehouse-304513.cars_ops_postgres_public.blockers_events`
  WHERE blocking_reason = 'pickup_too_soon'
    AND name = 'blocker_created'
    AND time >= '{DATE_FROM}' AND time < '{DATE_TO}'
  GROUP BY car_id
),
car_status_at_pts AS (
  SELECT p.car_id, p.pts_time, ce.name AS status_event,
    ROW_NUMBER() OVER (PARTITION BY p.car_id ORDER BY ce.time DESC) AS rn
  FROM pts_events p
  INNER JOIN `datawarehouse-304513.cars_ops_postgres_public.car_events` ce
    ON p.car_id = ce.car_id
  WHERE ce.name LIKE 'car_state_changed_%'
    AND ce.time <= p.pts_time
)
SELECT car_id, pts_time, status_event
FROM car_status_at_pts
WHERE rn = 1
  AND status_event IN (
    'car_state_changed_ready_to_deliver',
    'car_state_changed_tech_prep_done',
    'car_state_changed_arrived_from_supplier',
    'car_state_changed_produced',
    'car_state_changed_returned'
  )
"""
infleeting_pts_df = run_query(infleeting_pts_query)

# Also get total (unfiltered) PTS count to show the noise ratio
total_all_pts = run_query(f"""
  SELECT COUNT(DISTINCT car_id) AS n
  FROM `datawarehouse-304513.cars_ops_postgres_public.blockers_events`
  WHERE blocking_reason = 'pickup_too_soon'
    AND name = 'blocker_created'
    AND time >= '{DATE_FROM}' AND time < '{DATE_TO}'
""").iloc[0]['n']

print(f'Total PTS cars (all statuses): {total_all_pts}')
print(f'Infleeting PTS cars: {len(infleeting_pts_df)} ({100*len(infleeting_pts_df)/total_all_pts:.1f}%)')
print(f'Return/in-subscription PTS (noise): {total_all_pts - len(infleeting_pts_df)} ({100*(total_all_pts - len(infleeting_pts_df))/total_all_pts:.1f}%)')
print()
print('Status breakdown of infleeting PTS cars:')
print(infleeting_pts_df['status_event'].value_counts().to_string())

In [ ]:
# Step 2: Co-active blockers for infleeting PTS cars
# Use JOIN approach to avoid correlated subquery issues
coactive_query = f"""
WITH
pts_events AS (
  SELECT car_id, MIN(time) AS pts_time
  FROM `datawarehouse-304513.cars_ops_postgres_public.blockers_events`
  WHERE blocking_reason = 'pickup_too_soon'
    AND name = 'blocker_created'
    AND time >= '{DATE_FROM}' AND time < '{DATE_TO}'
  GROUP BY car_id
),
car_status_at_pts AS (
  SELECT p.car_id, p.pts_time, ce.name AS status_event,
    ROW_NUMBER() OVER (PARTITION BY p.car_id ORDER BY ce.time DESC) AS rn
  FROM pts_events p
  INNER JOIN `datawarehouse-304513.cars_ops_postgres_public.car_events` ce
    ON p.car_id = ce.car_id
  WHERE ce.name LIKE 'car_state_changed_%' AND ce.time <= p.pts_time
),
infleeting_pts AS (
  SELECT car_id, pts_time FROM car_status_at_pts
  WHERE rn = 1 AND status_event IN (
    'car_state_changed_ready_to_deliver','car_state_changed_tech_prep_done',
    'car_state_changed_arrived_from_supplier','car_state_changed_produced',
    'car_state_changed_returned')
),
blocker_created AS (
  SELECT car_id, blocking_reason, time AS created_time
  FROM `datawarehouse-304513.cars_ops_postgres_public.blockers_events`
  WHERE name = 'blocker_created'
    AND blocking_reason NOT IN ('pickup_too_soon','pickup_date_in_the_past','handover_date_in_the_past','return_not_scheduled')
),
blocker_resolved AS (
  SELECT car_id, blocking_reason, time AS resolved_time
  FROM `datawarehouse-304513.cars_ops_postgres_public.blockers_events`
  WHERE name IN ('blocker_resolved','blocker_deleted')
    AND blocking_reason NOT IN ('pickup_too_soon','pickup_date_in_the_past','handover_date_in_the_past','return_not_scheduled')
),
coactive_check AS (
  SELECT p.car_id, p.pts_time, bc.blocking_reason, bc.created_time,
    DATE_DIFF(DATE(p.pts_time), DATE(bc.created_time), DAY) AS days_active,
    MIN(br.resolved_time) AS first_resolution
  FROM infleeting_pts p
  INNER JOIN blocker_created bc ON p.car_id = bc.car_id AND bc.created_time < p.pts_time
  LEFT JOIN blocker_resolved br ON bc.car_id = br.car_id
    AND bc.blocking_reason = br.blocking_reason
    AND br.resolved_time > bc.created_time AND br.resolved_time <= p.pts_time
  GROUP BY p.car_id, p.pts_time, bc.blocking_reason, bc.created_time
)
SELECT
  blocking_reason,
  COUNT(DISTINCT car_id) AS cars,
  COUNT(*) AS episodes,
  ROUND(APPROX_QUANTILES(days_active, 100)[OFFSET(50)], 0) AS median_days_active,
  ROUND(APPROX_QUANTILES(days_active, 100)[OFFSET(75)], 0) AS p75_days_active
FROM coactive_check
WHERE first_resolution IS NULL  -- still active when PTS fired
GROUP BY blocking_reason
ORDER BY cars DESC
"""
coactive_df = run_query(coactive_query)

total_infleeting = len(infleeting_pts_df)
cars_with_coactive = coactive_df['cars'].max() if len(coactive_df) > 0 else 0
# Actually need distinct count across all blockers
cars_with_any = run_query(f"""
WITH
pts_events AS (
  SELECT car_id, MIN(time) AS pts_time
  FROM `datawarehouse-304513.cars_ops_postgres_public.blockers_events`
  WHERE blocking_reason = 'pickup_too_soon' AND name = 'blocker_created'
    AND time >= '{DATE_FROM}' AND time < '{DATE_TO}'
  GROUP BY car_id
),
car_status_at_pts AS (
  SELECT p.car_id, p.pts_time, ce.name AS status_event,
    ROW_NUMBER() OVER (PARTITION BY p.car_id ORDER BY ce.time DESC) AS rn
  FROM pts_events p
  INNER JOIN `datawarehouse-304513.cars_ops_postgres_public.car_events` ce
    ON p.car_id = ce.car_id
  WHERE ce.name LIKE 'car_state_changed_%' AND ce.time <= p.pts_time
),
infleeting_pts AS (
  SELECT car_id, pts_time FROM car_status_at_pts
  WHERE rn = 1 AND status_event IN ('car_state_changed_ready_to_deliver','car_state_changed_tech_prep_done','car_state_changed_arrived_from_supplier','car_state_changed_produced','car_state_changed_returned')
),
bc AS (
  SELECT car_id, blocking_reason, time AS ct FROM `datawarehouse-304513.cars_ops_postgres_public.blockers_events`
  WHERE name = 'blocker_created' AND blocking_reason NOT IN ('pickup_too_soon','pickup_date_in_the_past','handover_date_in_the_past','return_not_scheduled')
),
br AS (
  SELECT car_id, blocking_reason, time AS rt FROM `datawarehouse-304513.cars_ops_postgres_public.blockers_events`
  WHERE name IN ('blocker_resolved','blocker_deleted') AND blocking_reason NOT IN ('pickup_too_soon','pickup_date_in_the_past','handover_date_in_the_past','return_not_scheduled')
),
chk AS (
  SELECT p.car_id, bc.blocking_reason, bc.ct, MIN(br.rt) AS fr
  FROM infleeting_pts p
  INNER JOIN bc ON p.car_id = bc.car_id AND bc.ct < p.pts_time
  LEFT JOIN br ON bc.car_id = br.car_id AND bc.blocking_reason = br.blocking_reason AND br.rt > bc.ct AND br.rt <= p.pts_time
  GROUP BY p.car_id, bc.blocking_reason, bc.ct, p.pts_time
)
SELECT COUNT(DISTINCT car_id) AS n FROM chk WHERE fr IS NULL
""").iloc[0]['n']

coactive_df['pct'] = round(100 * coactive_df['cars'] / total_infleeting, 1)

print(f'=== INFLEETING PTS: Co-Active Blockers ({DATE_FROM} to {DATE_TO}) ===')
print(f'Infleeting PTS cars: {total_infleeting}')
print(f'With at least one co-active blocker: {cars_with_any} ({100*cars_with_any/total_infleeting:.1f}%)')
print(f'Without co-active blocker: {total_infleeting - cars_with_any} ({100*(total_infleeting - cars_with_any)/total_infleeting:.1f}%)')
print()
print(f'{"Blocker":<35} {"Cars":>6} {"%":>6} {"Med days":>9} {"P75":>6}')
print('-' * 65)
for _, row in coactive_df.iterrows():
    print(f"{row['blocking_reason']:<35} {row['cars']:>6} {row['pct']:>5.1f}% {row['median_days_active']:>9.0f} {row['p75_days_active']:>6.0f}")

# Chart
if len(coactive_df) > 0:
    fig = px.bar(
        coactive_df.head(15), y='blocking_reason', x='cars', orientation='h',
        title=f'Infleeting PTS: Co-Active Blockers ({DATE_FROM} to {DATE_TO}, n={total_infleeting})',
        labels={'cars': 'Cars Affected', 'blocking_reason': 'Blocker'},
        color='median_days_active', color_continuous_scale='YlOrRd', text='pct'
    )
    fig.update_traces(texttemplate='%{text}%', textposition='outside')
    fig.update_layout(height=500, width=900, yaxis={'categoryorder': 'total ascending'})
    fig.update_coloraxes(colorbar_title='Median Days<br>Active')
    fig.show()

## 2. The "No Co-Active Blocker" Cars — What Happened?

~33% of infleeting PTS cars had no other blocker active when PTS fired.
Did they never have blockers, or did blockers resolve just before PTS?

In [ ]:
# For cars WITHOUT co-active blockers: when was the last blocker resolved? Did they get shifted?
no_coactive_query = f"""
WITH
pts_events AS (
  SELECT car_id, MIN(time) AS pts_time
  FROM `datawarehouse-304513.cars_ops_postgres_public.blockers_events`
  WHERE blocking_reason = 'pickup_too_soon' AND name = 'blocker_created'
    AND time >= '{DATE_FROM}' AND time < '{DATE_TO}'
  GROUP BY car_id
),
car_status_at_pts AS (
  SELECT p.car_id, p.pts_time, ce.name AS status_event,
    ROW_NUMBER() OVER (PARTITION BY p.car_id ORDER BY ce.time DESC) AS rn
  FROM pts_events p
  INNER JOIN `datawarehouse-304513.cars_ops_postgres_public.car_events` ce
    ON p.car_id = ce.car_id
  WHERE ce.name LIKE 'car_state_changed_%' AND ce.time <= p.pts_time
),
infleeting_pts AS (
  SELECT car_id, pts_time FROM car_status_at_pts
  WHERE rn = 1 AND status_event IN ('car_state_changed_ready_to_deliver','car_state_changed_tech_prep_done','car_state_changed_arrived_from_supplier','car_state_changed_produced','car_state_changed_returned')
),
bc AS (
  SELECT car_id, blocking_reason, time AS ct FROM `datawarehouse-304513.cars_ops_postgres_public.blockers_events`
  WHERE name = 'blocker_created' AND blocking_reason NOT IN ('pickup_too_soon','pickup_date_in_the_past','handover_date_in_the_past','return_not_scheduled')
),
br AS (
  SELECT car_id, blocking_reason, time AS rt FROM `datawarehouse-304513.cars_ops_postgres_public.blockers_events`
  WHERE name IN ('blocker_resolved','blocker_deleted') AND blocking_reason NOT IN ('pickup_too_soon','pickup_date_in_the_past','handover_date_in_the_past','return_not_scheduled')
),
coactive AS (
  SELECT p.car_id, bc.blocking_reason, bc.ct, MIN(br.rt) AS fr
  FROM infleeting_pts p
  INNER JOIN bc ON p.car_id = bc.car_id AND bc.ct < p.pts_time
  LEFT JOIN br ON bc.car_id = br.car_id AND bc.blocking_reason = br.blocking_reason AND br.rt > bc.ct AND br.rt <= p.pts_time
  GROUP BY p.car_id, bc.blocking_reason, bc.ct, p.pts_time
),
cars_with_coactive AS (SELECT DISTINCT car_id FROM coactive WHERE fr IS NULL),
no_coactive_cars AS (
  SELECT p.car_id, p.pts_time FROM infleeting_pts p
  LEFT JOIN cars_with_coactive c ON p.car_id = c.car_id WHERE c.car_id IS NULL
),
-- Last resolved blocker before PTS
last_resolved AS (
  SELECT n.car_id, br.blocking_reason, br.rt AS resolved_time,
    DATE_DIFF(DATE(n.pts_time), DATE(br.rt), DAY) AS days_gap,
    ROW_NUMBER() OVER (PARTITION BY n.car_id ORDER BY br.rt DESC) AS rn
  FROM no_coactive_cars n
  INNER JOIN br ON n.car_id = br.car_id AND br.rt < n.pts_time
),
-- Shift within 7d of PTS
shifted AS (
  SELECT DISTINCT n.car_id
  FROM no_coactive_cars n
  INNER JOIN `datawarehouse-304513.cars_ops_postgres_public.car_events` ce
    ON n.car_id = ce.car_id
  WHERE ce.name = 'car_reservation_handover_shifted'
    AND ce.time >= n.pts_time AND DATE_DIFF(DATE(ce.time), DATE(n.pts_time), DAY) <= 7
)
SELECT
  (SELECT COUNT(*) FROM no_coactive_cars) AS total_no_coactive,
  (SELECT COUNT(*) FROM no_coactive_cars n WHERE NOT EXISTS (SELECT 1 FROM br WHERE br.car_id = n.car_id AND br.rt < n.pts_time)) AS never_had_blocker,
  (SELECT COUNT(*) FROM last_resolved WHERE rn = 1 AND days_gap <= 1) AS resolved_1d,
  (SELECT COUNT(*) FROM last_resolved WHERE rn = 1 AND days_gap <= 7) AS resolved_7d,
  (SELECT COUNT(*) FROM last_resolved WHERE rn = 1 AND days_gap <= 30) AS resolved_30d,
  (SELECT COUNT(*) FROM shifted) AS got_shifted
"""
nc_df = run_query(no_coactive_query)

t = nc_df['total_no_coactive'].iloc[0]
print(f'=== Cars with PTS but NO co-active blocker: {t} ===')
print(f'Never had any blocker resolved before PTS: {nc_df["never_had_blocker"].iloc[0]}')
print(f'Last blocker resolved within 1d of PTS: {nc_df["resolved_1d"].iloc[0]}')
print(f'Last blocker resolved within 7d of PTS: {nc_df["resolved_7d"].iloc[0]}')
print(f'Last blocker resolved within 30d of PTS: {nc_df["resolved_30d"].iloc[0]}')
print(f'Got handover SHIFTED within 7d: {nc_df["got_shifted"].iloc[0]} ({100*nc_df["got_shifted"].iloc[0]/t:.0f}%)')
print()
print('Pattern: blockers resolve at the last minute, but timeline is already too tight.')
print('The car technically becomes "ready" but pickup date is imminent -> PTS fires -> shift.')

In [ ]:
## 3. Blockers Active at Ready-to-Deliver Transition

The lifecycle state machine transitions to RTD when `TECH_PREP_DONE` + `REGISTERED`.
It does **not** check blockers. This means cars get customer-facing dates while
blockers like `sa_number_missing`, `pdi_not_done`, or `tires_not_mounted` are still active.

**Why is RTD set so loosely?**

```
register-cars lambda (scheduled)
  -> updates registration to REGISTERED
  -> calls updateLifecycleState()
  -> TECH_PREP_DONE + REGISTERED? -> READY_TO_DELIVER
  -> customer gets fixed handover date
  (NO blocker checks in this chain)
```

## 4. Co-Active Blockers by Brand/Compound (Infleeting Only)

In [ ]:
# Compound heatmap - filter to infleeting PTS cars only
compound_query = f"""
WITH
pts_events AS (
  SELECT car_id, MIN(time) AS pts_time
  FROM `datawarehouse-304513.cars_ops_postgres_public.blockers_events`
  WHERE blocking_reason = 'pickup_too_soon' AND name = 'blocker_created'
    AND time >= '{DATE_FROM}' AND time < '{DATE_TO}'
  GROUP BY car_id
),
car_status_at_pts AS (
  SELECT p.car_id, p.pts_time, ce.name AS status_event,
    ROW_NUMBER() OVER (PARTITION BY p.car_id ORDER BY ce.time DESC) AS rn
  FROM pts_events p
  INNER JOIN `datawarehouse-304513.cars_ops_postgres_public.car_events` ce
    ON p.car_id = ce.car_id
  WHERE ce.name LIKE 'car_state_changed_%' AND ce.time <= p.pts_time
),
infleeting_pts AS (
  SELECT car_id, pts_time FROM car_status_at_pts
  WHERE rn = 1 AND status_event IN ('car_state_changed_ready_to_deliver','car_state_changed_tech_prep_done','car_state_changed_arrived_from_supplier','car_state_changed_produced','car_state_changed_returned')
),
pts_with_brand AS (
  SELECT DISTINCT p.car_id, p.pts_time,
    JSON_VALUE(ce.snapshot, '$.oem') AS brand,
    COALESCE(JSON_VALUE(ce.snapshot, '$.infleeting_compound_id'), JSON_VALUE(ce.snapshot, '$.compound_id')) AS compound
  FROM infleeting_pts p
  INNER JOIN `datawarehouse-304513.cars_ops_postgres_public.car_events` ce
    ON p.car_id = ce.car_id AND ce.name = 'car_state_changed_ready_to_deliver'
),
bc AS (
  SELECT car_id, blocking_reason, time AS ct FROM `datawarehouse-304513.cars_ops_postgres_public.blockers_events`
  WHERE name = 'blocker_created' AND blocking_reason NOT IN ('pickup_too_soon','pickup_date_in_the_past','handover_date_in_the_past','return_not_scheduled')
),
br AS (
  SELECT car_id, blocking_reason, time AS rt FROM `datawarehouse-304513.cars_ops_postgres_public.blockers_events`
  WHERE name IN ('blocker_resolved','blocker_deleted') AND blocking_reason NOT IN ('pickup_too_soon','pickup_date_in_the_past','handover_date_in_the_past','return_not_scheduled')
),
coactive AS (
  SELECT p.car_id, p.brand, p.compound, bc.blocking_reason, bc.ct,
    MIN(br.rt) AS fr
  FROM pts_with_brand p
  INNER JOIN bc ON p.car_id = bc.car_id AND bc.ct < p.pts_time
  LEFT JOIN br ON bc.car_id = br.car_id AND bc.blocking_reason = br.blocking_reason AND br.rt > bc.ct AND br.rt <= p.pts_time
  GROUP BY p.car_id, p.brand, p.compound, bc.blocking_reason, bc.ct, p.pts_time
)
SELECT brand, compound, blocking_reason, COUNT(DISTINCT car_id) AS cars
FROM coactive WHERE fr IS NULL
GROUP BY brand, compound, blocking_reason
ORDER BY cars DESC
"""
compound_df = run_query(compound_query)

if len(compound_df) > 0:
    compound_df['label'] = compound_df['brand'].fillna('?') + ' @ ' + compound_df['compound'].fillna('?').str.replace('_', ' ').str.title()

    print(f'Top 20 compound/blocker combos (infleeting PTS, {DATE_FROM} to {DATE_TO}):')
    print(f'{"Brand @ Compound":<45} {"Blocker":<35} {"Cars":>5}')
    print('-' * 88)
    for _, row in compound_df.head(20).iterrows():
        print(f"{row['label']:<45} {row['blocking_reason']:<35} {row['cars']:>5}")

    top_compounds = compound_df.groupby('label')['cars'].sum().nlargest(15).index
    top_blockers = compound_df.groupby('blocking_reason')['cars'].sum().nlargest(10).index
    heat = compound_df[
        compound_df['label'].isin(top_compounds) & compound_df['blocking_reason'].isin(top_blockers)
    ].pivot_table(index='label', columns='blocking_reason', values='cars', fill_value=0)
    heat = heat.loc[heat.sum(axis=1).sort_values(ascending=False).index]
    heat = heat[heat.sum().sort_values(ascending=False).index]

    fig = px.imshow(
        heat, color_continuous_scale='YlOrRd',
        labels=dict(x='Co-Active Blocker', y='Brand @ Compound', color='Cars'),
        title=f'Infleeting PTS: Co-Active Blockers by Compound ({DATE_FROM} to {DATE_TO})',
        aspect='auto', text_auto=True
    )
    fig.update_layout(height=max(500, len(heat) * 30), width=1000)
    fig.update_xaxes(tickangle=45)
    fig.show()

## 5. Summary

### Two Different `pickup_too_soon` Systems

| System | Source | Cars affected | Nature |
|--------|--------|--------------|--------|
| **Infleeting** | `check-infleeting-blockers.ts` | ~1,100/year | Car not ready for customer handover |
| **Return** | `check-return-blockers.ts` | ~20,000/year | Return pickup within 24h (transient) |

Querying without filtering by car status mixes these populations and gives misleading numbers.

### Infleeting PTS Breakdown (2025 full year, verified)

- **1,094 infleeting PTS cars**
- **67% had a co-active blocker** when PTS fired (sa_number, pdi, tires, deposit...)
- **33% had no co-active blocker** — but most had blockers resolve within days before PTS
  - ~50% of these had a blocker resolve within 7 days of PTS
  - ~66% still got their handover shifted despite "no active blocker"
  - Pattern: blocker resolves at last minute, but timeline already too tight

### Why RTD is Set Too Loosely

The lifecycle state machine (`update-car-lifecycle-state.ts:122-124`) only checks:
- `TECH_PREP_DONE` + `REGISTERED` -> `READY_TO_DELIVER`

It does **not** consult:
- Active blockers (sa_number, pdi, tires, deposit, etc.)
- Whether the car is physically at the correct compound
- Whether a delivery ride exists

This means a car can get a **customer-facing handover date** while critical prep steps
are still incomplete. The blocker system runs separately and only fires PTS reactively
when the handover is already imminent.

### The Root Cause Chain

```
1. Car reaches TECH_PREP_DONE + REGISTERED
2. Auto-transition to READY_TO_DELIVER (no blocker check)
3. Customer gets fixed handover date
4. Blockers still active (sa_number, pdi, tires...)
5. Time passes, no one acts on non-critical blockers
6. Handover approaches -> blocker becomes CRITICAL -> auto-shift
7. Customer delay (or: blocker resolves last-minute but timeline too tight -> PTS -> shift)
```